learning how to flow

In [ ]:
%load_ext autoreload
%autoreload 2

# IMPORTS
import os
import time
import tqdm
import math
import pandas as pd
from datetime import datetime

import torch
from torch import nn
import numpy as np
# import torchdyn
from torchdyn.core import NeuralODE 


from data import * #only here bc you can plot the data for the main notebook, but is generallly used in the training loop inside flow_matching.py
from dpd import *
from model_architecture import * 
from transformer import *
# from utils import *
from flow_matching import * 
from discrete_fm import *


In [3]:
####### DEVICE SETUP #######
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("device is", device)

device is mps


In [4]:
def set_seed(seed):
    torch.manual_seed(seed)   # seeds CPU, CUDA and MPS generators
    np.random.seed(seed)      # insurance; your code uses default_rng, so this is optional

### Data & Sampling


**data.py**

Sample from the distributions (Cat or Moon)  
Data sits on -4 to 4.0 grid in the X-Y axis space  

**def sample_cats(N=1500):**  
&emsp;Samples N points from the cat distribution.  
&emsp;Return:  
&emsp;&emsp;numpy array of each xy coordinate in the sampled dist

**def sample_moons(N=1500):**  
&emsp;Uses the torchdfyn module to generate N samples of the moon distribution  
&emsp;Return:  
&emsp;&emsp;torch.tensor of each xy coordinate in the sampled dist. 

**def plot_distribution(sampled_points[list], dist_name[list],color[list]):**  
&emsp;Plots the sampled points, dist_name and color are lists of strings  

In [5]:
# ############ UNCOMMENT TO PLOT DISTRIBUTIONS ############
# #sample and plot cats
# sampled_cats = sample_cats(1500)
# # plot_distribution([sampled_cats], ["Cats"], ["gold"])
# #sample and plot moons
# sampled_moons = sample_moons(1500)
# # plot_distribution([sampled_moons], ["Moon"], ["steelblue"])
# #plot both together
# plot_distribution([sampled_moons, sampled_cats], ["Moon", "Cats"], ["steelblue", "gold"])

**dpd.py**

Generates some type of source distribution (symmetric or binomial) and applies a discrete process to it (poisson) to generate the target distribution

**def make_measure_specs(n_measures=None, centers=None, min_center=4, max_center=50, seed=None, ** spec_kwargs ** )**

&emsp;Uses n_measures or centers to define the number of sources/where the center of the source is.
For each defined measure, it creates the corresponding DiscreteMeasure object which defines the attributes of the data: center, sample type, process type, timesteps, half width.

&emsp;Return:  
&emsp;&emsp;a list of DiscreteMeasure objects

In [6]:
SEED = 10
N = 1500
rng = np.random.default_rng(SEED)
CENTERS = [20]
N_MEASURES = 2
#initialize the data object using the list of centers
spec = make_measure_specs(centers=CENTERS, seed=SEED)

#initialize the data object using the list of centers
# spec = make_measure_specs(n_measures=N_MEASURES, seed=SEED)

#create the source and target according to the spec parameters, then plot
for s in spec:
    #processed is the timesteps of the discrete process
    processed = generate_target_dist(spec=s, sample_size=N, rng=rng)

    # plot_dpd(processed, sample_type=s.sample_type, process_type=s.process_type, center=s.center)

    animate_discrete_histograms(
        processed,                         # (T, N) trajectory: rows are timesteps
        V=int(processed.max()) + 1,        # number of bins, must cover the largest count
        target=processed[-1],              # last timestep = target distribution
        center=s.center,                   # dashed line at the source center
        title_prefix=f"{s.process_type.upper()} Process applied to a {s.sample_type.upper()} Distribution (center {s.center})"
        )

### Discrete Flow Matching

Set up model

In [7]:
%%time

# key is the dataset name, value is the dimension
data_dict = {"dpd":1,
             "catmoons": 2}
data = "dpd" #"catmoons" "dpd"
dim = data_dict[data]
V = 128
#set up the model and optimizer
# model = MLP(dim=dim, out_dim = dim * V, w=512, time_varying=True).to(device)
model = DiscreteAttentionFlow(
            D=dim,
            out_dim = dim * V,
            num_hidden=64,
            num_layers=3,
            num_heads=1,
            num_ffn_layers=2,
            num_conditions=None,
            activation=nn.GELU,
            dropout=0.05,
        ).to(device)


# optimizer = torch.optim.Adam(model.parameters())
# print(f"model on device: {device}")

# dfm = DiscreteFlowMatching(optimizer, device, model, V, "catmoons")

#training loop
# dfm.train(epochs=20000, batch_size = 256)

128 outdim
CPU times: user 5.72 ms, sys: 7.1 ms, total: 12.8 ms
Wall time: 35.8 ms


/Users/magui/miniconda3/lib/python3.12/site-packages/torch/nn/init.py:511: UserWarning: Initializing zero-element tensors is a no-op
  warnings.warn("Initializing zero-element tensors is a no-op")


Model training for discrete data

In [ ]:
"""SET UP PARAMETERS"""
SEED = 10
N = 1500
rng = np.random.default_rng(SEED)

"""PLOT TEST DATA"""
# test centers are between [30, 50]
TEST_CENTERS = [30, 35, 40, 45, 50]
#initialize the data object using the list of centers
spec_test = make_measure_specs(centers=TEST_CENTERS, seed=SEED)

#create the source and target according to the spec parameters
for s in spec_test:
    #processed is the timesteps of the discrete process
    processed_test = generate_target_dist(spec=s, sample_size=N, rng=rng)

    # plot_dpd(processed, sample_type=s.sample_type, process_type=s.process_type, center=s.center)

    animate_discrete_histograms(
        processed_test,                         # (T, N) trajectory: rows are timesteps
        V=int(processed_test.max()) + 1,        # number of bins, must cover the largest count
        target=processed_test[-1],              # last timestep = target distribution
        center=s.center,                   # dashed line at the source center
        title_prefix=f"{s.process_type.upper()} Process applied to a {s.sample_type.upper()} Distribution (center {s.center})"
        )
    
    break

In [1]:
# """
# - 1, 5, 10, 15, 20, 25? train measures
# - train 0-19, test 20-40
# """
# """MAKE TRAIN MEASURES"""
# TRAIN_N_MEASURES = 1 #ablate using 1, 5, 10, 15, 20, 25 measures
# #initialize the data object using the list of centers
# spec_train = make_measure_specs(n_measures=TRAIN_N_MEASURES, seed=SEED)

# print(f"SEED {SEED}, TRAINING {TRAIN_N_MEASURES} MEASURES")

# dpd_train = MeasureDataset(spec_train, seed=SEED)
# assert  type(dpd_train) == MeasureDataset, f"not a MeasureDataset object its a {type(dpd_train)}"

# print()

# optimizer = torch.optim.Adam(model.parameters())
# print(f"model on device: {device}")

# """TRAIN MODEL"""
# dfm = DiscreteFlowMatching(optimizer, device, model, V, data, dataset=dpd_train)

# # training loop
# dfm.train(epochs=20000, batch_size = 256)





In [ ]:
######SETTING UP PARAMETERS#########
TRAIN_SEED = 10
TEST_SEED = 1010
# key is the dataset name, value is the dimension
data_dict = {"dpd":1,
             "catmoons": 2}
data = "dpd" #"catmoons" "dpd"
dim = data_dict[data]
V = 128
####################################

all_results = []
############
#make test set
dpd_test = MeasureDataset(spec_test, seed=TEST_SEED)
############

for world_seed in [0]:
    #set up csv path saving
    stamp = datetime.now().strftime("%Y%m%d_%H%M") # once per ablation
    csv_path = f"results/ablation_training_measures/world_seed{world_seed}/trainseed{TRAIN_SEED}_testseed{TEST_SEED}_{stamp}.csv"
    os.makedirs(os.path.dirname(csv_path), exist_ok=True)

    #ablate number of training measures
    for n in [1, 5, 10, 15, 20, 25]:
        #reset world seed
        set_seed(world_seed)


        spec_train = make_measure_specs(n_measures=n, seed=TRAIN_SEED)
        dpd_train = MeasureDataset(spec_train, seed=TRAIN_SEED)

        print(f"WORLD_SEED {world_seed}, TRAIN_SEED {TRAIN_SEED}, TRAINING {n} MEASURES")

        model = DiscreteAttentionFlow(
                D=dim,
                out_dim = dim * V,
                num_hidden=64,
                num_layers=3,
                num_heads=1,
                num_ffn_layers=2,
                num_conditions=None,
                activation=nn.GELU,
                dropout=0.05,
            ).to(device)
        optimizer = torch.optim.Adam(model.parameters())

        dfm = DiscreteFlowMatching(optimizer, device, model, V, data, dataset=dpd_train)
        dfm.train(epochs=20000, batch_size=256, save_gif=True, display_anim=False)


        all_results += dfm.evaluate(dpd_test, N=1024)
        #save results immediately
        pd.DataFrame(all_results).to_csv(csv_path, index=False)

        break

In [ ]:
dpd_test = MeasureDataset(spec_test, seed=SEED+1000)

test_results = dfm.evaluate(dpd_test, N=1024)

df_test = pd.DataFrame(test_results).set_index("center")



,W1,energy,W1_floor,energy_floor
center,,,,
20,6.100,3.154,0.261,0.007
25,1.294,0.198,0.466,0.021
30,5.371,2.998,0.312,0.013
35,10.210,11.330,0.410,0.021
40,15.364,21.098,0.385,0.021


mean over test measures:
W1              7.668
energy          7.756
W1_floor        0.367
energy_floor    0.017
dtype: float64


### OG Conditional Flow Matching

In [ ]:
%%time
#2D distribution so dim is 2
dim = 2
#set up the model and optimizer
model = MLP(dim=dim, time_varying=True).to(device)
optimizer = torch.optim.Adam(model.parameters())
print(f"model on device: {device}")

fm = FlowMatching(optimizer, device, model)
#training loop
fm.train()